Scientific source notebook for the isolated reproduction tree. Calculation cells and saved execution objects are preserved. Narrative navigation has been adapted for this package. Use the reproduction runner and frozen-input prerequisites before executing. Historical evaluation blocks remain reused evidence; this copy does not establish independent replication.


# 09 · Portfolio covariance evaluation

**Question:** Does the thematic factor improve the risk estimate of a collection of holdings?

Both models are refitted on identical complete training dates across all fixed holdings. Predicted variance combines factor covariance and either diagonal or within-industry residual covariance. Future missing quotes omit dates, never holdings.

Research report · Method and code guide · Evidence map


**Chapter route**

Measurement and interpretation principles · Data and design context · preserve the paired comparison · Read and run this notebook · Calculation source: covariance evaluation · Calculation source: reporting · Calculation source: publication · Results, examples and interpretation · Worked example: variance underestimation and QLIKE · Visual evidence · Findings · Analysis and takeaways · Bridge through the study


## Measurement and interpretation principles

Portfolio variance combines factor covariance and residual covariance: $\widehat{\Sigma}=B\widehat{\Omega}B^\top+\widehat{\Psi}$ and $\widehat v_p=w^\top\widehat{\Sigma}w$. Annualize the daily variance by 252. Compare diagonal and within-industry residual structures using identical holdings, complete training dates and covariance denominator $n-1$. Formation-fixed target weights imply daily rebalancing; missing future quotes remove dates, never holdings. QLIKE is $v/\widehat v-\log(v/\widehat v)-1$, with lower loss preferred.


## Data and design context

The test specification was frozen in `docs/TEST_FREEZE.md` before this evaluation. Portfolio memberships are fixed at formation from eligible securities, snapshot weights, company size quartiles and the pre-specified SMH-loading median split. Both share classes are retained.

Each portfolio uses common complete training dates across every holding and all seven factor series. Both models are refitted on exactly those dates. The minimum remains 400 observations for the 504-session window or 200 for the 252-session sensitivity. A failed common-date requirement excludes the entire comparison, never a retrospectively selected holding.

Predicted covariance combines sample factor covariance with either diagonal residual covariance or residual covariance within GICS industry groups. Every covariance uses denominator n−1, so differences do not arise from different degrees-of-freedom corrections. Future returns use fixed target weights, with an entire date omitted if any holding or required factor is missing. At least 40 common future dates are required. Only complete factor quarters through 2026 Q2 are evaluated.

QLIKE is realized/predicted − log(realized/predicted) − 1. Negative paired SMH-minus-baseline QLIKE differences favor the extension. Newey–West uncertainty and a moving-block bootstrap use actual quarterly spacing, retaining missing-quarter gaps. These are conditional on estimated loadings and portfolio memberships; the first stage is not resampled. Six confirmation quarters offer limited independent evidence.

The saved membership, date omissions, training coverage, daily realized returns and quarterly evaluations support every summary. Size-dependent files remain under `results/licensed/covariance_evaluation`, with licensed-source provenance. Rebuild using `python -I -m quiet_index_500.covariance_evaluation` in the virtual environment.

The original noncircular block intervals are retained, with circular moving-block intervals added as a reviewer-requested sensitivity. Circular starts give every calendar quarter equal expected inclusion, addressing boundary imbalance in a six-quarter block. Missing quarter slots remain missing under modulo calendar indexing, and counts of rejected all-missing bootstrap draws are saved. This changes uncertainty diagnostics only; portfolios, samples and point estimates remain fixed.


## preserve the paired comparison

`formation_portfolios` fixes membership and normalized target weights. `predict_portfolio_variances` refits both models on common dates with covariance denominator n−1. `realized_portfolio_variance` retains target weights when evaluating future common dates. `variance_losses` measures QLIKE = realized/predicted − log(realized/predicted) − 1.


## Read and run this notebook

The calculation definitions now live here, alongside the purpose and interpretation. Use the chapter route for the argument and the bold function names for individual definitions. Definition cells do not download data or run the full study. The results/examples follow the definitions. Save edits and restart the kernel before a full rerun so shared imports read the saved version.


## Calculation source: covariance evaluation

These are the actual definitions used by the study. Read the purpose above, then follow the functions below before the results. The small Python adapter imports only these marked cells; it does not run collection or display cells.


### Imports, conventions and measurement settings

Imports and fixed conventions used by the calculations below.


In [1]:
"""Formation-fixed portfolio variance evaluation with paired model observations.

Every portfolio is selected before inspecting its future returns. Both factor
models are refitted to the same common complete training dates for every holding
in that portfolio. Factor and residual covariance use denominator n-1, so adding a
factor cannot gain or lose variance solely through a changed degrees correction.
"""

import sys

import numpy as np
import pandas as pd

from quiet_index_500 import factor_exposures, settings
from quiet_index_500.factor_exposures import standard_factor_names
from quiet_index_500.foundation import calendar, loader
from quiet_index_500.research_io import save_table


**formation_portfolios**

Return predetermined portfolio membership and normalized fixed target weights.

Securities are factor-eligible at formation; both share classes are retained.
Size membership is assigned once per issuer using the saved company quartile,
then mapped to all its eligible share classes. Size portfolios and SMH halves
are snapshot-weighted; the equal portfolio is equal-security-weighted.


In [2]:
def formation_portfolios(exposures, identifiers, size_rows):
    """Return predetermined portfolio membership and normalized fixed target weights.

    Securities are factor-eligible at formation; both share classes are retained.
    Size membership is assigned once per issuer using the saved company quartile,
    then mapped to all its eligible share classes. Size portfolios and SMH halves
    are snapshot-weighted; the equal portfolio is equal-security-weighted.
    """
    eligible = exposures.loc[exposures["eligible"]].set_index("ticker")
    members = identifiers.loc[eligible.index].copy()
    members["smh_loading"] = eligible["m4_smh_excess"]
    quartile_by_cik = size_rows.set_index("cik")["size_quartile"]
    if not quartile_by_cik.index.is_unique:
        raise ValueError("size rows must contain one formation and one row per issuer")
    members["size_quartile"] = members["cik"].map(quartile_by_cik)
    median_loading = members["smh_loading"].median()
    assignments = {"snapshot": members.index, "equal": members.index,
                   "high_smh": members.index[members["smh_loading"] > median_loading],
                   "low_smh": members.index[members["smh_loading"] <= median_loading]}
    assignments.update({f"size_q{quartile}": members.index[members["size_quartile"] == quartile] for quartile in range(1, 5)})
    rows = []
    for portfolio, tickers in assignments.items():
        selected = members.loc[tickers]
        unnormalized = pd.Series(1.0, index=tickers) if portfolio == "equal" else selected["snapshot_weight"]
        weights = unnormalized / unnormalized.sum()
        for ticker in tickers:
            rows.append({"portfolio": portfolio, "ticker": ticker, "cik": int(selected.at[ticker, "cik"]),
                         "weight": float(weights[ticker]), "snapshot_weight": float(selected.at[ticker, "snapshot_weight"]),
                         "gics_industry_group_code": selected.at[ticker, "gics_industry_group_code"],
                         "formation_smh_loading": float(selected.at[ticker, "smh_loading"]),
                         "smh_split_median": float(median_loading) if portfolio.endswith("smh") else np.nan,
                         "licensed_size_dependency": portfolio.startswith("size_")})
    return pd.DataFrame(rows)


**predict_portfolio_variances**

Fit the two models on one common matrix and return annualized predictions.

The caller supplies fixed weights indexed by ticker. Missing dates are omitted
jointly; no holding is removed. The result contains the exact training dates,
training omission diagnostics and predictions for both residual treatments.


In [3]:
def predict_portfolio_variances(training_excess_returns, training_factors, weights, industry_groups, minimum_observations):
    """Fit the two models on one common matrix and return annualized predictions.

    The caller supplies fixed weights indexed by ticker. Missing dates are omitted
    jointly; no holding is removed. The result contains the exact training dates,
    training omission diagnostics and predictions for both residual treatments.
    """
    stocks = training_excess_returns.reindex(columns=weights.index)
    factor_names = [*standard_factor_names, "smh_excess"]
    factors = training_factors.reindex(stocks.index)[factor_names]
    common = stocks.notna().all(axis=1) & factors.notna().all(axis=1)
    if not np.isclose(weights.sum(), 1.0, atol=1e-12):
        raise ValueError("fixed portfolio weights must sum to one")
    omissions = pd.DataFrame({"date": stocks.index[~common],
                              "missing_security_count": stocks.isna().sum(axis=1).loc[~common].to_numpy(),
                              "missing_factor_count": factors.isna().sum(axis=1).loc[~common].to_numpy()})
    observation_dates = stocks.index[common]
    if len(observation_dates) < minimum_observations:
        return {"eligible": False, "reason": "insufficient_common_training_dates", "observations": len(observation_dates),
                "observation_dates": observation_dates, "omissions": omissions, "predictions": pd.DataFrame()}
    response = stocks.loc[common].to_numpy(dtype=float)
    ordered_weights = weights.to_numpy(dtype=float)
    industry = industry_groups.reindex(weights.index)
    if industry.isna().any():
        raise ValueError("industry labels required for the fixed holdings")
    predictions = []
    for model, names in (("ff5_mom", standard_factor_names), ("ff5_mom_smh", factor_names)):
        design = np.column_stack([np.ones(len(response)), factors.loc[common, list(names)].to_numpy(dtype=float)])
        coefficients, _, rank, _ = np.linalg.lstsq(design, response, rcond=None)
        if rank != design.shape[1]:
            raise ValueError("rank-deficient covariance factor design")
        fitted = design @ coefficients
        residual = response - fitted
        fitted_portfolio = fitted @ ordered_weights
        factor_variance = np.var(fitted_portfolio, ddof=1)
        diagonal_residual_variance = float((ordered_weights ** 2) @ np.var(residual, axis=0, ddof=1))
        industry_residual_variance = 0.0
        for group in industry.unique():
            selection = industry.to_numpy() == group
            weighted_group_residual = residual[:, selection] @ ordered_weights[selection]
            industry_residual_variance += float(np.var(weighted_group_residual, ddof=1))
        for residual_treatment, residual_variance in (("diagonal", diagonal_residual_variance), ("industry_block", industry_residual_variance)):
            predictions.append({"model": model, "residual_treatment": residual_treatment,
                                "predicted_variance": (factor_variance + residual_variance) * settings.annualisation_sessions,
                                "factor_variance": factor_variance * settings.annualisation_sessions,
                                "residual_variance": residual_variance * settings.annualisation_sessions,
                                "training_observations": len(response), "factor_design_rank": rank,
                                "covariance_denominator": len(response) - 1})
    return {"eligible": True, "reason": "eligible", "observations": len(response), "observation_dates": observation_dates,
            "omissions": omissions, "predictions": pd.DataFrame(predictions)}


**realized_portfolio_variance**

Realized annualized variance with fixed weights and no missing-quote reweighting.


In [4]:
def realized_portfolio_variance(future_excess_returns, future_factors, weights, minimum_observations):
    """Realized annualized variance with fixed weights and no missing-quote reweighting."""
    stocks = future_excess_returns.reindex(columns=weights.index)
    factors = future_factors.reindex(stocks.index)[[*standard_factor_names, "smh_excess"]]
    common = stocks.notna().all(axis=1) & factors.notna().all(axis=1)
    omissions = pd.DataFrame({"date": stocks.index[~common],
                              "missing_security_count": stocks.isna().sum(axis=1).loc[~common].to_numpy(),
                              "missing_factor_count": factors.isna().sum(axis=1).loc[~common].to_numpy()})
    portfolio_returns = stocks.loc[common] @ weights
    eligible = len(portfolio_returns) >= minimum_observations
    return {"eligible": eligible, "reason": "eligible" if eligible else "insufficient_common_future_dates",
            "observations": len(portfolio_returns),
            "realized_variance": float(portfolio_returns.var(ddof=1) * settings.annualisation_sessions) if eligible else np.nan,
            "returns": portfolio_returns, "omissions": omissions}


**variance_losses**

Scale-free variance diagnostics; QLIKE is r/p-log(r/p)-1, lower is better.


In [5]:
def variance_losses(predicted_variance, realized_variance):
    """Scale-free variance diagnostics; QLIKE is r/p-log(r/p)-1, lower is better."""
    if predicted_variance <= 0 or realized_variance <= 0:
        raise ValueError("variance loss requires positive predicted and realized variance")
    bias_ratio = predicted_variance / realized_variance
    inverse_ratio = 1 / bias_ratio
    log_ratio = float(np.log(bias_ratio))
    return {"bias_ratio": bias_ratio, "log_variance_ratio": log_ratio,
            "absolute_log_variance_ratio": abs(log_ratio),
            "qlike": inverse_ratio - np.log(inverse_ratio) - 1,
            "underestimated": predicted_variance < realized_variance}


**_block_resample_indices**

This helper supports the calculation sequence below; its arguments and return value are shown in the code.


In [6]:
def _block_resample_indices(starts, block_length, calendar_count, circular=False):
    indices = (starts[:, :, None] + np.arange(block_length)).reshape(len(starts), -1)[:, :calendar_count]
    return indices % calendar_count if circular else indices


**_valid_resample_means**

This helper supports the calculation sequence below; its arguments and return value are shown in the code.


In [7]:
def _valid_resample_means(dense_values, indices):
    samples = dense_values[indices]
    finite_counts = np.isfinite(samples).sum(axis=1)
    accepted = finite_counts > 0
    means = np.nansum(samples[accepted], axis=1) / finite_counts[accepted]
    if len(means) == 0:
        raise ValueError("all bootstrap samples contain only missing quarters")
    return means, int((~accepted).sum())


**quarterly_mean_uncertainty**

Quarter-level NW and moving-block mean uncertainty, conditional on estimates.

Resampling units are whole quarters, preserving common shocks inside each
portfolio. The bootstrap does not reestimate first-stage loadings or memberships.


In [8]:
def quarterly_mean_uncertainty(values, *, quarters=None, replications=settings.bootstrap_replications):
    """Quarter-level NW and moving-block mean uncertainty, conditional on estimates.

    Resampling units are whole quarters, preserving common shocks inside each
    portfolio. The bootstrap does not reestimate first-stage loadings or memberships.
    """
    values = np.asarray(values, dtype=float)
    if not np.isfinite(values).all() or len(values) == 0:
        raise ValueError("nonempty finite quarterly differences required")
    count = len(values)
    mean = float(values.mean())
    if quarters is None:
        dense_values = values
    else:
        periods = pd.PeriodIndex(pd.to_datetime(quarters), freq="Q")
        if not periods.is_unique or len(periods) != count:
            raise ValueError("one value per unique calendar quarter required")
        dense_values = pd.Series(values, index=periods).reindex(pd.period_range(periods.min(), periods.max(), freq="Q")).to_numpy()
    calendar_count = len(dense_values)
    centered = np.where(np.isfinite(dense_values), dense_values - mean, 0.0)  # missing quarters contribute no pair, never collapse a lag
    lag_count = min(settings.newey_west_lags, calendar_count - 1)
    long_run_variance = float(centered @ centered / count)
    for lag in range(1, lag_count + 1):
        autocovariance = float(centered[lag:] @ centered[:-lag] / count)
        long_run_variance += 2 * (1 - lag / (lag_count + 1)) * autocovariance
    standard_error = np.sqrt(max(long_run_variance, 0) / count)
    block_length = min(settings.bootstrap_block_length, calendar_count)
    generator = np.random.default_rng(settings.bootstrap_random_seed)
    blocks_needed = int(np.ceil(calendar_count / block_length))
    starts = generator.integers(0, calendar_count - block_length + 1, size=(replications, blocks_needed))
    indices = _block_resample_indices(starts, block_length, calendar_count)
    bootstrap_means, all_missing = _valid_resample_means(dense_values, indices)
    circular_generator = np.random.default_rng(settings.bootstrap_random_seed)
    circular_starts = circular_generator.integers(0, calendar_count, size=(replications, blocks_needed))
    circular_indices = _block_resample_indices(circular_starts, block_length, calendar_count, circular=True)
    circular_means, circular_all_missing = _valid_resample_means(dense_values, circular_indices)
    return {"quarters": count, "calendar_quarters": calendar_count, "missing_quarters": calendar_count - count,
            "mean_difference": mean, "newey_west_standard_error": standard_error,
            "newey_west_lags": lag_count, "normal_interval_low": mean - 1.96 * standard_error,
            "normal_interval_high": mean + 1.96 * standard_error,
            "bootstrap_interval_low": float(np.quantile(bootstrap_means, 0.025)),
            "bootstrap_interval_high": float(np.quantile(bootstrap_means, 0.975)),
            "bootstrap_block_length": block_length, "bootstrap_replications": replications,
            "bootstrap_valid_replications": len(bootstrap_means), "bootstrap_all_missing_replications": all_missing,
            "circular_bootstrap_interval_low": float(np.quantile(circular_means, 0.025)),
            "circular_bootstrap_interval_high": float(np.quantile(circular_means, 0.975)),
            "circular_bootstrap_valid_replications": len(circular_means),
            "circular_bootstrap_all_missing_replications": circular_all_missing}


**summarize_covariance_results**

Period summaries and paired SMH-minus-baseline loss differences by portfolio.


In [9]:
def summarize_covariance_results(evaluations):
    """Period summaries and paired SMH-minus-baseline loss differences by portfolio."""
    valid = evaluations.loc[evaluations["evaluation_eligible"]].copy()
    summaries, differences, inference = [], [], []
    group_columns = ["window_sessions", "portfolio", "residual_treatment"]
    for keys, group in valid.groupby(group_columns, sort=True):
        identity = dict(zip(group_columns, keys))
        for block in ("all", "development", "validation", "confirmation"):
            period = group if block == "all" else group.loc[group["outcome_block"] == block]
            if period.empty:
                continue
            for model, model_rows in period.groupby("model"):
                summaries.append({**identity, "outcome_block": block, "model": model, "quarters": len(model_rows),
                                  "mean_bias_ratio": model_rows["bias_ratio"].mean(),
                                  "mean_log_variance_ratio": model_rows["log_variance_ratio"].mean(),
                                  "mean_absolute_log_variance_ratio": model_rows["absolute_log_variance_ratio"].mean(),
                                  "mean_qlike": model_rows["qlike"].mean(),
                                  "share_underestimated": model_rows["underestimated"].mean(),
                                  "mean_training_observations": model_rows["training_observations"].mean(),
                                  "mean_future_observations": model_rows["future_observations"].mean()})
            for metric in ("qlike", "absolute_log_variance_ratio", "log_variance_ratio"):
                paired = period.pivot(index="formation_date", columns="model", values=metric).dropna()
                if paired.empty:
                    continue
                differences_by_date = paired["ff5_mom_smh"] - paired["ff5_mom"]
                if block == "all":
                    for date, difference in differences_by_date.items():
                        differences.append({**identity, "formation_date": date, "metric": metric, "smh_minus_baseline": difference})
                inference.append({**identity, "outcome_block": block, "metric": metric,
                                  **quarterly_mean_uncertainty(differences_by_date.to_numpy(), quarters=differences_by_date.index)})
    return {"period_summary": pd.DataFrame(summaries), "paired_quarterly_differences": pd.DataFrame(differences),
            "paired_inference": pd.DataFrame(inference)}


**build_covariance_evaluation**

Run the frozen portfolio comparisons for both windows and save exact evidence.


In [10]:
def build_covariance_evaluation():
    """Run the frozen portfolio comparisons for both windows and save exact evidence."""
    freeze_path = settings.repository_root_folder / "docs/TEST_FREEZE.md"
    if not freeze_path.exists():
        raise RuntimeError("freeze the tests before evaluating outcomes")
    identifiers = loader.load_identifiers()
    price_panels = loader.load_price_panels()
    daily_returns = loader.daily_simple_returns_from_prices(price_panels.adjusted_close, price_panels.quarantined_returns)
    factors = loader.load_factors()
    excess_returns = loader.excess_returns(daily_returns, factors["rf"])
    common_factors = factors[list(standard_factor_names)].join(loader.load_smh_excess_returns().rename("smh_excess"))
    size_path = settings.licensed_results_folder / "size_liquidity/size.csv"
    size = pd.read_csv(size_path, parse_dates=["formation_date"])
    sessions = daily_returns.index
    factor_cutoff = loader.factor_data_last_date()
    evaluation_rows, membership_rows, omission_rows, realized_rows, training_rows = [], [], [], [], []
    exposure_paths = []
    for window_sessions, minimum in ((settings.main_window_sessions, settings.main_window_minimum_observations),
                                     (settings.sensitivity_window_sessions, settings.sensitivity_window_minimum_observations)):
        exposure_path = settings.results_folder / "factor_exposures" / f"exposures_{window_sessions}.csv"
        exposure_paths.append(exposure_path)
        exposures = pd.read_csv(exposure_path, parse_dates=["formation_date"])
        for formation_date, formation_exposures in exposures.groupby("formation_date", sort=True):
            cutoff = calendar.measurement_cutoff(formation_date, factor_cutoff)
            training_window = calendar.estimation_window(sessions, cutoff, window_sessions)
            future_window = calendar.following_quarter_sessions(sessions, formation_date)
            complete_outcome = calendar.following_quarter_complete(formation_date, factor_cutoff)
            memberships = formation_portfolios(formation_exposures, identifiers, size.loc[size["formation_date"] == formation_date])
            memberships["formation_date"] = formation_date
            memberships["window_sessions"] = window_sessions
            membership_rows.append(memberships)
            for portfolio, member_rows in memberships.groupby("portfolio", sort=True):
                weights = member_rows.set_index("ticker")["weight"]
                industry = identifiers["gics_industry_group_code"]
                common = {"formation_date": formation_date, "measurement_cutoff": cutoff, "window_sessions": window_sessions,
                          "portfolio": portfolio, "holdings": len(weights), "issuers": member_rows["cik"].nunique(),
                          "represented_snapshot_weight": member_rows["snapshot_weight"].sum(),
                          "outcome_quarter": str(calendar.outcome_quarter(formation_date)), "outcome_block": calendar.outcome_block(formation_date),
                          "outcome_complete": complete_outcome, "licensed_size_dependency": portfolio.startswith("size_")}
                predicted = predict_portfolio_variances(excess_returns.reindex(training_window), common_factors, weights, industry, minimum)
                training_rows.append({**common, "training_window_sessions": len(training_window), "training_observations": predicted["observations"],
                                      "training_omitted_dates": len(predicted["omissions"]), "training_eligible": predicted["eligible"],
                                      "training_reason": predicted["reason"], "minimum_observations": minimum,
                                      "window_first_session": training_window.min(), "window_last_session": training_window.max()})
                for row in predicted["omissions"].to_dict("records"):
                    omission_rows.append({**common, "phase": "training", **row})
                future = realized_portfolio_variance(excess_returns.reindex(future_window), common_factors, weights,
                                                     settings.forward_outcome_minimum_valid_days) if complete_outcome else None
                if future is not None:
                    for row in future["omissions"].to_dict("records"):
                        omission_rows.append({**common, "phase": "future", **row})
                    for date, value in future["returns"].items():
                        realized_rows.append({**common, "date": date, "portfolio_excess_return": value})
                evaluation_eligible = predicted["eligible"] and complete_outcome and future["eligible"]
                reason = (predicted["reason"] if not predicted["eligible"] else "incomplete_factor_outcome" if not complete_outcome
                          else future["reason"])
                prediction_rows = predicted["predictions"].to_dict("records") if predicted["eligible"] else [
                    {"model": model, "residual_treatment": treatment, "predicted_variance": np.nan,
                     "training_observations": predicted["observations"]}
                    for model in ("ff5_mom", "ff5_mom_smh") for treatment in ("diagonal", "industry_block")]
                for prediction in prediction_rows:
                    row = {**common, **prediction, "evaluation_eligible": bool(evaluation_eligible), "evaluation_reason": reason,
                           "training_omitted_dates": len(predicted["omissions"]), "future_sessions": len(future_window),
                           "future_observations": future["observations"] if future is not None else 0,
                           "future_omitted_dates": len(future["omissions"]) if future is not None else len(future_window),
                           "realized_variance": future["realized_variance"] if future is not None else np.nan}
                    if evaluation_eligible:
                        row.update(variance_losses(row["predicted_variance"], row["realized_variance"]))
                    evaluation_rows.append(row)
            print(f"covariance: {window_sessions} sessions, formation {formation_date.date()}", flush=True)
    tables = {"quarterly_evaluation": pd.DataFrame(evaluation_rows), "membership": pd.concat(membership_rows, ignore_index=True),
              "date_omissions": pd.DataFrame(omission_rows), "realized_daily_returns": pd.DataFrame(realized_rows),
              "training_coverage": pd.DataFrame(training_rows)}
    tables.update(summarize_covariance_results(tables["quarterly_evaluation"]))
    outputs = {}
    for name, table in tables.items():
        size_rows = table["portfolio"].str.startswith("size_")
        for licensed, selected in ((False, table.loc[~size_rows]), (True, table.loc[size_rows])):
            inputs = [freeze_path, *exposure_paths, *([size_path] if licensed else [])]
            path = save_table(selected, "covariance_evaluation", f"{name}.csv", licensed=licensed, inputs=inputs,
                              source_modules=[settings.repository_root_folder / 'src/quiet_index_500/covariance_evaluation.py', factor_exposures],
                              note="Formation-fixed holdings and target weights; both models refit on the same portfolio-specific complete training dates. "
                                   "Factor and residual covariances use n-1. Future common dates exclude any missing holding or factor; no holding reweighting. "
                                   "QLIKE is realized/predicted-log(realized/predicted)-1. Quarter uncertainty conditions on memberships and estimated loadings.")
            outputs[f"{'size_' if licensed else ''}{name}"] = path
    return outputs


**saved_covariance_summary**

Read public and size-dependent period summaries for the executed notebook.


In [11]:
def saved_covariance_summary():
    """Read public and size-dependent period summaries for the executed notebook."""
    return pd.concat([pd.read_csv(folder / "covariance_evaluation/period_summary.csv")
                      for folder in (settings.results_folder, settings.licensed_results_folder)], ignore_index=True)


## Calculation source: reporting

These are the actual definitions used by the study. Read the purpose above, then follow the functions below before the results. The small Python adapter imports only these marked cells; it does not run collection or display cells.


### Imports, conventions and measurement settings

Imports and fixed conventions used by the calculations below.


In [12]:
"""Compact report evidence and scientific figures from saved evaluation tables."""

import sys

import pandas as pd

from quiet_index_500 import settings
from quiet_index_500.research_io import save_table


**build_report_tables_and_figures**

Select and label the statistics cited by the final research report.


In [13]:
def build_report_tables_and_figures():
    """Select and label the statistics cited by the final research report."""
    forward_folder = settings.licensed_results_folder / "forward_tests"
    inference_path = forward_folder / "paired_inference.csv"
    coefficient_path = forward_folder / "coefficient_inference.csv"
    inference = pd.read_csv(inference_path)
    forecasts = inference[inference.analysis.eq("forecast") & inference.metric.eq("relative_mse_improvement")].copy()
    for column in ["mean", "bootstrap_low", "bootstrap_high"]:
        forecasts[f"{column}_percent"] = forecasts[column] * 100
    forecasts["positive_means"] = "SMH/extension reduces MSE relative to its paired restricted model"
    paths = {"forecast_summary": save_table(forecasts, "report", "forecast_summary.csv", licensed=True,
              inputs=[inference_path], source_modules=[settings.repository_root_folder / 'src/quiet_index_500/reporting.py'],
              note="Quarter-equal percentage MSE improvement; negative is worse. Intervals conditional on first-stage estimates; confirmation has only six quarters.")}
    coefficients = pd.read_csv(coefficient_path)
    paths["association_summary"] = save_table(coefficients, "report", "association_summary.csv", licensed=True,
                                               inputs=[coefficient_path], source_modules=[settings.repository_root_folder / 'src/quiet_index_500/reporting.py'])
    covariance_paths = [folder / "covariance_evaluation/period_summary.csv" for folder in [settings.results_folder, settings.licensed_results_folder]]
    covariance = pd.concat([pd.read_csv(path) for path in covariance_paths], ignore_index=True)
    key = ["window_sessions", "portfolio", "residual_treatment", "outcome_block"]
    baseline = covariance[covariance.model.eq("ff5_mom")].drop(columns="model")
    augmented = covariance[covariance.model.eq("ff5_mom_smh")].drop(columns="model")
    paired = baseline.merge(augmented, on=key, suffixes=("_baseline", "_smh"), validate="one_to_one")
    paired["qlike_improvement_percent"] = 100 * (1 - paired.mean_qlike_smh / paired.mean_qlike_baseline)
    paired["positive_means"] = "SMH reduces mean QLIKE relative to baseline"
    paths["covariance_summary"] = save_table(paired, "report", "covariance_summary.csv", licensed=True,
                                               inputs=covariance_paths, source_modules=[settings.repository_root_folder / 'src/quiet_index_500/reporting.py'])
    _figures(forecasts, paired)
    return paths


**_figures**

This helper supports the calculation sequence below; its arguments and return value are shown in the code.


In [14]:
def _figures(forecasts, covariance):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import numpy as np
    destination = settings.results_folder / "report"
    destination.mkdir(exist_ok=True)
    periods = ["development", "validation", "confirmation"]
    labels = {"semiconductor": "Signed SMH, sector controls", "semiconductor_industry": "Signed SMH, industry controls",
              "semiconductor_variance_component": "SMH variance component (exploratory)"}
    fig, axis = plt.subplots(figsize=(10, 5))
    subset = forecasts[forecasts.window_sessions.eq(504) & forecasts.outcome.eq("log_forward_residual_volatility")]
    for index, (name, label) in enumerate(labels.items()):
        values = subset[subset.comparison.eq(name)].set_index("outcome_block").reindex(periods)
        axis.bar(np.arange(3) + (index - 1) * .24, values.mean_percent, width=.24, label=label)
    axis.axhline(0, color="black", linewidth=.8)
    axis.set_xticks(np.arange(3), ["Development\n12 forecast quarters", "Validation\n8 quarters", "Confirmation\n6 quarters"])
    axis.set(ylabel="Mean quarterly reduction in log-risk MSE (%)", title="SMH does not consistently improve residual-risk forecasts")
    axis.legend(fontsize=8)
    fig.text(.01, .01, "504-session exposures; expanding training; matched companies. Positive is better. Six confirmation quarters limit inference.", fontsize=8)
    fig.tight_layout(rect=(0, .04, 1, 1))
    fig.savefig(destination / "forward_residual_risk.png", dpi=160)
    plt.close(fig)
    fig, axis = plt.subplots(figsize=(10, 5))
    subset = covariance[covariance.window_sessions.eq(504) & covariance.portfolio.eq("snapshot")]
    for index, (name, label) in enumerate([("diagonal", "Diagonal residual covariance"), ("industry_block", "Within-industry residual covariance")]):
        values = subset[subset.residual_treatment.eq(name)].set_index("outcome_block").reindex(periods)
        axis.bar(np.arange(3) + (index - .5) * .32, values.qlike_improvement_percent, width=.32, label=label)
    axis.axhline(0, color="black", linewidth=.8)
    axis.set_xticks(np.arange(3), ["Development\n19 eligible quarters", "Validation\n8 quarters", "Confirmation\n6 quarters"])
    axis.set(ylabel="Reduction in mean QLIKE (%)", title="Snapshot portfolio: SMH benefit depends on residual treatment")
    axis.legend(fontsize=8)
    fig.text(.01, .01, "Same fixed holdings, common training/future dates, and covariance denominator. Positive is better; FMP size not used for this portfolio.", fontsize=8)
    fig.tight_layout(rect=(0, .04, 1, 1))
    fig.savefig(destination / "snapshot_covariance.png", dpi=160)
    plt.close(fig)


## Calculation source: publication

These are the actual definitions used by the study. Read the purpose above, then follow the functions below before the results. The small Python adapter imports only these marked cells; it does not run collection or display cells.


### Imports, conventions and measurement settings

Imports and fixed conventions used by the calculations below.


In [15]:
"""Rebuild publication figures and reading tables from saved study results, offline.

This presentation layer does not estimate models or load raw vendor data. Each
figure has its own CSV and manifest. Run after quiet_index_500.reproduce.
"""
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
from quiet_index_500 import settings
from quiet_index_500.foundation.manifests import write_manifest, sha256_of_file

ROOT = settings.repository_root_folder
OUT = ROOT / "results/publication"
PERIODS = ["development", "validation", "confirmation"]
COLORS = ["#176B87", "#D16B31", "#467B57", "#845A97"]


**read**

This helper supports the calculation sequence below; its arguments and return value are shown in the code.


In [16]:
def read(relative):
    return pd.read_csv(ROOT / "results" / relative)


**save_data**

This helper supports the calculation sequence below; its arguments and return value are shown in the code.


In [17]:
def save_data(name, frame, sources, note):
    OUT.mkdir(parents=True, exist_ok=True)
    path = OUT / f"{name}.csv"
    frame.to_csv(path, index=False, float_format="%.12g")
    write_manifest(path, [ROOT / "results" / s for s in sources],
                   "quiet_index_500.publication", source_modules=[settings.repository_root_folder / 'src/quiet_index_500/publication.py'], note=note)
    return path


**save_figure**

This helper supports the calculation sequence below; its arguments and return value are shown in the code.


In [18]:
def save_figure(fig, name, subtitle):
    import matplotlib.pyplot as plt
    fig.text(.06, .015, subtitle, fontsize=9, color="#53616b", va="bottom")
    fig.tight_layout(rect=(0, .08, 1, .98))
    for extension in ("png", "svg"):
        fig.savefig(OUT / f"{name}.{extension}", dpi=220, facecolor="white")
    plt.close(fig)


**build_publication**

This helper supports the calculation sequence below; its arguments and return value are shown in the code.


In [19]:
def build_publication():
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 11,
                        "axes.titlesize": 15, "axes.titleweight": "bold",
                        "axes.spines.top": False, "axes.spines.right": False,
                        "axes.labelcolor": "#253544", "text.color": "#253544",
                        "axes.prop_cycle": plt.cycler(color=COLORS), "svg.fonttype": "none"})

    # 1. Report companies and represented snapshot weight, not just panel rows.
    coverage = read("licensed/panel/coverage_504.csv")
    labels = {"m4_smh_excess": "Factor exposure", "log_market_cap": "Company size",
              "log_amihud_252_per_million": "Amihud illiquidity", "own_momentum": "Own momentum"}
    data = coverage[coverage.characteristic.isin(labels)].copy()
    save_data("01_coverage", data, ["licensed/panel/coverage_504.csv"],
              "Company-level valid measurements, 504-session panel; snapshot weight is retrospective, not historical market coverage.")
    fig, axes = plt.subplots(2, 1, figsize=(10.4, 7), sharex=True)
    for name, label in labels.items():
        rows = data[data.characteristic.eq(name)]
        dates = pd.to_datetime(rows.formation_date)
        axes[0].plot(dates, rows.companies, label=label, linewidth=2)
        axes[1].plot(dates, rows.represented_snapshot_weight * 100, linewidth=2)
    axes[0].axhline(500, color="#8b979e", ls=":", label="500-company snapshot")
    axes[0].set(title="The company panel is complete; each measure has its own sample", ylabel="Companies with a value", ylim=(0, 520))
    axes[0].legend(ncol=2, fontsize=9, loc="lower right")
    axes[1].set(ylabel="Represented snapshot weight (%)", ylim=(0, 103), xlabel="Formation date")
    save_figure(fig, "01_coverage", "36 formations; one representative per issuer. September 2026 factor measurements stop at August 31.\nSource: Notebook 07, combining 04–06. Later regressions apply their own matched-sample rules.")

    # 2. Distribution and persistence answer distinct questions.
    panel = read("licensed/panel/company_panel_504.csv")
    quantiles = panel.groupby("formation_date").m4_smh_excess.quantile([.1, .5, .9]).unstack()
    quantiles.columns = ["p10", "p50", "p90"]
    quantiles = quantiles.reset_index()
    stability = read("licensed/panel/loading_stability_504.csv")
    data = quantiles.merge(stability, on="formation_date", how="left", validate="one_to_one")
    data["measurement_cutoff"] = pd.to_datetime(data.formation_date).clip(upper=pd.Timestamp("2026-08-31"))
    save_data("02_exposure_stability", data, ["licensed/panel/company_panel_504.csv", "licensed/panel/loading_stability_504.csv"],
              "Equal-company loading quantiles and consecutive-formation Spearman rank correlation; persistence is not forecast evidence.")
    fig, axes = plt.subplots(2, 1, figsize=(10.4, 7), sharex=True)
    dates = pd.to_datetime(data.formation_date)
    axes[0].fill_between(dates, data.p10, data.p90, color=COLORS[0], alpha=.16, label="10th–90th percentile")
    axes[0].plot(dates, data.p50, linewidth=2, label="Median")
    axes[0].axhline(0, color="#8b979e", linewidth=.8)
    axes[0].set(title="Semiconductor loading varies across companies", ylabel="Conditional SMH loading")
    axes[0].legend(fontsize=9)
    axes[1].plot(dates, data.spearman, color=COLORS[2], linewidth=2)
    axes[1].set(ylabel="Consecutive-quarter rank correlation", ylim=(0, 1), xlabel="Formation date")
    save_figure(fig, "02_exposure_stability", "504-session estimates; paired companies for rank correlation. A stable exposure is not necessarily a useful forecast feature.\nSource: Notebook 07 using Notebook 04. Final factor cutoff: August 31, 2026.")

    # 3. A compact relationship map; save pairwise counts rather than imply one common sample.
    labels = {"log_market_cap": "Size (log cap)", "log_median_dollar_volume_60": "Dollar volume (log)",
              "log_amihud_252_per_million": "Amihud (log)", "ordinary_beta": "Ordinary beta",
              "downside_beta": "Downside beta", "own_momentum": "Own momentum", "m4_smh_excess": "SMH loading"}
    correlations = read("licensed/panel/correlations_504.csv")
    pairs = correlations[correlations["first"].isin(labels) & correlations["second"].isin(labels)]
    data = pairs.groupby(["first", "second"]).agg(mean_spearman=("spearman", "mean"),
                formations=("spearman", "count"), minimum_companies=("companies", "min"), maximum_companies=("companies", "max")).reset_index()
    save_data("03_characteristic_overlap", data, ["licensed/panel/correlations_504.csv"],
              "Equal-formation mean pairwise Spearman correlations; company samples differ by pair and formation. FMP-derived size included.")
    grid = pd.DataFrame(np.eye(len(labels)), index=labels, columns=labels)
    for row in data.itertuples():
        grid.loc[row.first, row.second] = grid.loc[row.second, row.first] = row.mean_spearman
    fig, ax = plt.subplots(figsize=(10.4, 8))
    im = ax.imshow(grid, vmin=-1, vmax=1, cmap="RdBu_r")
    ax.set_xticks(range(len(labels)), labels.values(), rotation=35, ha="right", fontsize=10)
    ax.set_yticks(range(len(labels)), labels.values(), fontsize=10)
    for i in range(len(labels)):
        for j in range(len(labels)):
            v = grid.iloc[i, j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=10, color="white" if abs(v)>.65 else "#253544")
    ax.set_title("Related characteristics are not interchangeable", pad=16)
    fig.colorbar(im, ax=ax, shrink=.7, label="Mean Spearman correlation")
    save_figure(fig, "03_characteristic_overlap", "36 equally weighted formations; pairwise eligible company samples; descriptive correlations, not causal effects.\nSource: Notebook 07, combining factor, downside, momentum, size and liquidity measurements.")

    # 4. Use corrected circular intervals explicitly; do not use old bootstrap_*_percent fields.
    forecasts = read("licensed/report/forecast_summary.csv")
    labels = {"semiconductor": "Signed SMH · sector controls", "semiconductor_industry": "Signed SMH · industry groups",
              "liquidity": "+ Amihud · separate extension", "dollar_volume": "+ Dollar volume · exploratory"}
    data = forecasts[(forecasts.window_sessions.eq(504)) & forecasts.outcome.eq("log_forward_residual_volatility")
                     & forecasts.comparison.isin(labels) & forecasts.outcome_block.isin(PERIODS[1:])].copy()
    for col in ["mean", "circular_bootstrap_low", "circular_bootstrap_high"]:
        data[col + "_pct"] = 100 * data[col]
    save_data("04_forward_risk", data, ["licensed/report/forecast_summary.csv"],
              "Mean quarterly percentage reduction in log-volatility MSE; 95% circular quarter-block intervals conditional on first-stage exposures. Liquidity comparisons add to the SMH model, not the styles-only baseline.")
    fig, axes = plt.subplots(1, 2, figsize=(12.2, 5), sharex=True, sharey=True)
    for ax, period in zip(axes, PERIODS[1:]):
        rows = data[data.outcome_block.eq(period)].set_index("comparison").reindex(labels)
        for i, (_, r) in enumerate(rows.iterrows()):
            ax.plot([r.circular_bootstrap_low_pct, r.circular_bootstrap_high_pct], [i, i], lw=2, color=COLORS[i])
            ax.scatter(r.mean_pct, i, s=55, color=COLORS[i], zorder=3)
            ax.annotate(f"{r.mean_pct:+.2f}%", (r.mean_pct, i), xytext=(0, 10), textcoords="offset points", ha="center", fontsize=9)
        ax.axvline(0, color="#8b979e", ls="--")
        ax.set_title(f"{period.title()} · {int(rows.quarters.iloc[0])} quarters")
        ax.set_xlabel("Reduction in log-risk forecast MSE (%)")
        ax.set_yticks(range(len(labels)), labels.values(), fontsize=10)
        ax.grid(axis="x", alpha=.15)
    axes[0].set_ylim(3.55, -.55)
    save_figure(fig, "04_forward_risk", "Positive = lower forecast error. Dots: quarter-equal means; lines: conditional 95% circular block intervals.\nNotebook 08: matched companies within each comparison. Only six confirmation quarters; exploratory results remain provisional.")

    # 5. Absolute losses retained in CSV; relative loss change has a different aggregation from Figure 4.
    covariance = read("licensed/report/covariance_summary.csv")
    labels = {"snapshot": "Snapshot weighted", "equal": "Equal security weighted", "high_smh": "Higher SMH loading half", "low_smh": "Lower SMH loading half"}
    data = covariance[covariance.window_sessions.eq(504) & covariance.outcome_block.eq("confirmation") & covariance.portfolio.isin(labels)].copy()
    save_data("05_portfolio_comparison", data, ["licensed/report/covariance_summary.csv"],
              "100*(1-mean QLIKE augmented/mean QLIKE baseline), six confirmation quarters per portfolio. Positive is better; portfolios overlap and are not independent tests.")
    fig, ax = plt.subplots(figsize=(10.4, 5.8))
    for i, (treatment, label) in enumerate([("diagonal", "Diagonal residuals"), ("industry_block", "Within-industry residuals")]):
        rows = data[data.residual_treatment.eq(treatment)].set_index("portfolio").reindex(labels)
        bars = ax.barh(np.arange(4)+(i-.5)*.32, rows.qlike_improvement_percent, height=.30, label=label, color=COLORS[i])
        ax.bar_label(bars, fmt="%+.2f%%", padding=4, fontsize=9)
    ax.axvline(0, color="#8b979e", linewidth=.8)
    ax.set_yticks(range(4), labels.values())
    ax.invert_yaxis()
    ax.set(xlabel="Reduction in period mean QLIKE (%)", title="Portfolio benefit depends on the residual-covariance baseline", xlim=(-6, 17))
    ax.legend(loc="lower right", fontsize=9)
    save_figure(fig, "05_portfolio_comparison", "2025 Q1–2026 Q2; 504-session estimation; fixed holdings and daily target weights. Positive = lower variance loss.\nNotebook 09. Six quarters per bar; point estimates only. Snapshot paired-difference intervals include zero under both treatments.")

    # 6. Variance levels explain what the loss comparisons are measuring.
    evaluation = read("covariance_evaluation/quarterly_evaluation.csv")
    data = evaluation[evaluation.window_sessions.eq(504) & evaluation.portfolio.eq("snapshot")].copy()
    save_data("06_variance_calibration", data, ["covariance_evaluation/quarterly_evaluation.csv"],
              "Annualized variance in decimal-return-squared units. Ineligible quarters remain in the table and plot as gaps; outcome quarters after 2026Q2 excluded from chart.")
    fig, axes = plt.subplots(2, 1, figsize=(10.4, 7.3), sharex=True)
    for ax, treatment, title in zip(axes, ["diagonal", "industry_block"], ["Diagonal residual covariance", "Within-industry residual covariance"]):
        rows = data[data.residual_treatment.eq(treatment) & data.outcome_complete].copy()
        rows["date"] = pd.PeriodIndex(rows.outcome_quarter, freq="Q").to_timestamp(how="end")
        for j, (model, label) in enumerate([("ff5_mom", "FF5 + momentum"), ("ff5_mom_smh", "+ SMH")]):
            r = rows[rows.model.eq(model)].sort_values("date")
            ax.plot(r.date, r.predicted_variance.where(r.evaluation_eligible), label=label, color=COLORS[j], lw=1.8)
        r = rows[rows.model.eq("ff5_mom")].sort_values("date")
        ax.plot(r.date, r.realized_variance.where(r.evaluation_eligible), color="#253544", label="Realized", lw=1.5, ls="--")
        ax.axvspan(pd.Timestamp("2025-01-01"), pd.Timestamp("2026-06-30"), color=COLORS[0], alpha=.06)
        ax.set(title=title, ylabel="Annualized variance (decimal²)", ylim=(0, None))
        ax.grid(axis="y", alpha=.15)
    axes[0].legend(ncol=3, fontsize=9)
    axes[1].set_xlabel("Following-quarter end")
    save_figure(fig, "06_variance_calibration", "Notebook 09: snapshot-weighted portfolio; 33 eligible outcome quarters. Gap = failed common training sample.\nShading marks six confirmation quarters. Realized variance is a noisy outcome measure; these are not return forecasts.")

    # 7. An event correction illustrates why the foundation matters to later estimates.
    data = read("review/event_sensitivity.csv")
    save_data("07_event_sensitivity", data, ["review/event_sensitivity.csv"],
              "DHR residual-volatility sensitivity; only three 504-session windows include the 2016 event. Alternative valuation changes no primary inputs.")
    fig, ax = plt.subplots(figsize=(10.4, 5.4))
    labels = {"original_vendor_double_adjustment": "Original vendor error", "primary_fortive_first_close": "Accepted correction", "form_8937_average_valuation": "Alternative distribution valuation"}
    for i, (scenario, label) in enumerate(labels.items()):
        r = data[data.scenario.eq(scenario)].sort_values("formation_date")
        ax.bar(np.arange(len(r))+(i-1)*.24, r.residual_volatility*100, width=.23, label=label, color=COLORS[i])
    ax.set_xticks(range(3), ["Dec 2017", "Mar 2018", "Jun 2018"])
    ax.set(ylabel="Annualized FF5 + momentum residual volatility (%)", xlabel="Formation date", title="One erroneous return can dominate a company's risk estimate")
    ax.legend(fontsize=9)
    save_figure(fig, "07_event_sensitivity", "DHR only; three affected 504-session windows. The two corrected series nearly overlap.\nFoundation correction → Notebook 04 estimates. Event scenarios change one return in memory, never raw data.")

    # Exact comparison samples and uncertainty used by the publication.
    quarters = read("licensed/forward_tests/quarters_504.csv")
    forward = quarters[quarters.analysis.eq("forecast") & quarters.outcome.eq("log_forward_residual_volatility")]
    counts = forward.groupby(["comparison", "outcome_block"]).agg(minimum_companies=("companies", "min"), maximum_companies=("companies", "max"),
                    first_formation=("formation_date", "min"), last_formation=("formation_date", "max")).reset_index()
    table = forecasts[forecasts.window_sessions.eq(504) & forecasts.outcome.eq("log_forward_residual_volatility") & forecasts.outcome_block.ne("all")].merge(counts, on=["comparison", "outcome_block"], validate="one_to_one")
    save_data("forecast_evidence", table, ["licensed/report/forecast_summary.csv", "licensed/forward_tests/quarters_504.csv"], "All main-window residual-risk forecast comparisons, with exact company-count ranges and conditional uncertainty.")
    ci = read("covariance_evaluation/paired_inference.csv")
    ci = ci[ci.window_sessions.eq(504) & ci.portfolio.eq("snapshot") & ci.metric.eq("qlike")]
    save_data("covariance_uncertainty", ci, ["covariance_evaluation/paired_inference.csv"], "Paired SMH-minus-baseline QLIKE; negative favors SMH. Use circular interval columns; six confirmation quarters.")
    render_report()
    files = sorted(p for p in OUT.iterdir() if p.suffix in {".png", ".svg", ".csv"})
    (OUT / "publication_manifest.json").write_text(json.dumps({"generator": "src/quiet_index_500/publication.py", "files": [
        {"file": p.relative_to(ROOT).as_posix(), "sha256": sha256_of_file(p)} for p in files]}, indent=2)+"\n")
    return OUT


**render_report**

Static local HTML; uses relative image links and no remote scripts/fonts.


In [20]:
def render_report():
    """Static local HTML; uses relative image links and no remote scripts/fonts."""
    source = ROOT / "docs/RESEARCH_REPORT.md"
    if not source.exists():
        return
    import mistune
    markdown = mistune.create_markdown(plugins=["table", "strikethrough", "url"])
    body = markdown(source.read_text())
    css = """body{font-family:system-ui,-apple-system,sans-serif;color:#253544;background:#f4f6f7;margin:0;line-height:1.65}
    main{max-width:1000px;margin:40px auto;padding:52px;background:white;border-top:6px solid #176B87}
    h1{font-size:2.5em;line-height:1.15;letter-spacing:-.035em}h2{margin-top:2em;line-height:1.25}h3{margin-top:1.6em}
    a{color:#176B87}img{max-width:100%;height:auto}table{border-collapse:collapse;font-size:.9em;width:100%;display:block;overflow:auto}
    th,td{border-bottom:1px solid #dce3e7;padding:10px;text-align:left}th{background:#edf3f5}blockquote{border-left:3px solid #176B87;margin-left:0;padding-left:20px;color:#53616b}
    code{font-family:ui-monospace,monospace;font-size:.85em}pre{overflow:auto;background:#f4f6f7;padding:16px}p,li{max-width:90ch}
    @media(max-width:700px){main{margin:0;padding:24px}h1{font-size:2em}}@media print{body{background:white}main{margin:0;padding:0;border:0}h2{break-after:avoid}img,table{break-inside:avoid}}"""
    (ROOT / "docs/RESEARCH_REPORT.html").write_text(f'<!doctype html><html lang="en"><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1"><title>Semiconductor exposure and subsequent risk</title><style>{css}</style><main>{body}</main></html>')


## Results, examples and interpretation

The cells below use the definitions above and the frozen study inputs.


In [21]:
summary = saved_covariance_summary()
summary

,window_sessions,portfolio,residual_treatment,outcome_block,model,quarters,mean_bias_ratio,mean_log_variance_ratio,mean_absolute_log_variance_ratio,mean_qlike,share_underestimated,mean_training_observations,mean_future_observations
0,252,equal,diagonal,all,ff5_mom,32,1.780225,0.133639,0.875872,1.102013,0.375000,240.343750,62.437500
1,252,equal,diagonal,all,ff5_mom_smh,32,1.781321,0.134290,0.876267,1.101510,0.375000,240.343750,62.437500
2,252,equal,diagonal,development,ff5_mom,18,1.770645,-0.041997,0.992965,1.639496,0.500000,235.611111,62.500000
3,252,equal,diagonal,development,ff5_mom_smh,18,1.770846,-0.041745,0.992957,1.638390,0.500000,235.611111,62.500000
4,252,equal,diagonal,validation,ff5_mom,8,1.833615,0.473485,0.569251,0.182266,0.125000,246.625000,62.750000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
251,504,size_q4,industry_block,development,ff5_mom_smh,20,1.551660,0.028537,0.799441,0.681125,0.500000,494.850000,62.850000
252,504,size_q4,industry_block,validation,ff5_mom,8,2.261043,0.742121,0.765088,0.265697,0.125000,504.000000,62.750000
253,504,size_q4,industry_block,validation,ff5_mom_smh,8,2.237719,0.730452,0.758631,0.261084,0.125000,504.000000,62.750000
254,504,size_q4,industry_block,confirmation,ff5_mom,6,1.737426,0.220340,0.815555,0.480034,0.333333,493.500000,62.166667


In [22]:
summary.loc[(summary['window_sessions'] == 504) & (summary['portfolio'] == 'snapshot')]

,window_sessions,portfolio,residual_treatment,outcome_block,model,quarters,mean_bias_ratio,mean_log_variance_ratio,mean_absolute_log_variance_ratio,mean_qlike,share_underestimated,mean_training_observations,mean_future_observations
112,504,snapshot,diagonal,all,ff5_mom,33,1.838263,0.222313,0.851875,0.634796,0.424242,452.454545,62.303030
113,504,snapshot,diagonal,all,ff5_mom_smh,33,1.856734,0.235110,0.850255,0.624358,0.424242,452.454545,62.303030
114,504,snapshot,diagonal,development,ff5_mom,19,1.652682,0.009448,0.878328,0.799983,0.578947,445.105263,62.263158
115,504,snapshot,diagonal,development,ff5_mom_smh,19,1.671619,0.025567,0.872076,0.781144,0.578947,445.105263,62.263158
116,504,snapshot,diagonal,validation,ff5_mom,8,2.351076,0.760380,0.776159,0.284815,0.125000,463.375000,62.750000
117,504,snapshot,diagonal,validation,ff5_mom_smh,8,2.376635,0.770844,0.784357,0.290052,0.125000,463.375000,62.750000
118,504,snapshot,diagonal,confirmation,ff5_mom,6,1.742183,0.178962,0.869059,0.578344,0.333333,461.166667,61.833333
119,504,snapshot,diagonal,confirmation,ff5_mom_smh,6,1.749730,0.184352,0.869022,0.573607,0.333333,461.166667,61.833333
120,504,snapshot,industry_block,all,ff5_mom,33,1.873210,0.244631,0.852181,0.620985,0.424242,452.454545,62.303030
121,504,snapshot,industry_block,all,ff5_mom_smh,33,1.860369,0.237060,0.850327,0.623517,0.424242,452.454545,62.303030


In [23]:
summary.loc[(summary['window_sessions'] == 504) & (summary['outcome_block'] == 'confirmation')]

,window_sessions,portfolio,residual_treatment,outcome_block,model,quarters,mean_bias_ratio,mean_log_variance_ratio,mean_absolute_log_variance_ratio,mean_qlike,share_underestimated,mean_training_observations,mean_future_observations
70,504,equal,diagonal,confirmation,ff5_mom,6,1.324799,0.010103,0.716647,0.621916,0.333333,461.166667,61.833333
71,504,equal,diagonal,confirmation,ff5_mom_smh,6,1.327195,0.011773,0.717696,0.621700,0.333333,461.166667,61.833333
78,504,equal,industry_block,confirmation,ff5_mom,6,1.356713,0.035987,0.721364,0.600486,0.333333,461.166667,61.833333
79,504,equal,industry_block,confirmation,ff5_mom_smh,6,1.354197,0.033620,0.721797,0.603565,0.333333,461.166667,61.833333
86,504,high_smh,diagonal,confirmation,ff5_mom,6,1.337715,-0.028684,0.680724,0.424129,0.333333,493.500000,62.166667
87,504,high_smh,diagonal,confirmation,ff5_mom_smh,6,1.427214,0.051626,0.678639,0.365974,0.333333,493.500000,62.166667
94,504,high_smh,industry_block,confirmation,ff5_mom,6,1.394884,0.021787,0.682515,0.388255,0.333333,493.500000,62.166667
95,504,high_smh,industry_block,confirmation,ff5_mom_smh,6,1.412878,0.038778,0.680561,0.375916,0.333333,493.500000,62.166667
102,504,low_smh,diagonal,confirmation,ff5_mom,6,1.300023,-0.014386,0.718289,0.676821,0.333333,461.500000,61.833333
103,504,low_smh,diagonal,confirmation,ff5_mom_smh,6,1.397616,0.060610,0.741128,0.623478,0.333333,461.500000,61.833333


## Worked example: variance underestimation and QLIKE

Illustrative inputs are separate from study data. Re-run this short cell to connect the method to its numerical output.


In [24]:
# Predict 20% annual volatility (variance .04), realize variance .08.
variance_losses(predicted_variance=.04, realized_variance=.08)

{'bias_ratio': 0.5,
 'log_variance_ratio': -0.6931471805599453,
 'absolute_log_variance_ratio': 0.6931471805599453,
 'qlike': np.float64(0.3068528194400546),
 'underestimated': True}

## Visual evidence

Source figure: 05_portfolio_comparison

Source figure: 06_variance_calibration

These figures use saved results; see the figure guide for sources, sample definitions and limitations.


## Findings

Snapshot confirmation QLIKE falls 0.82% under diagonal residuals but rises 3.13% with industry residuals. Both paired-difference circular intervals include zero. High/low-loading portfolios have confirmation gains but industry-aware validation deterioration.

## Analysis and takeaways

The snapshot portfolio's main-window confirmation mean QLIKE falls 0.82% with diagonal residuals and rises 3.13% with industry residuals. Both paired-difference circular intervals span zero. High- and low-loading halves improve in confirmation but deteriorate in industry-aware validation. These portfolio-dependent results do not establish a broad covariance advantage. Unlike 08's mean quarterly percentage MSE reductions, these percentages compare period mean QLIKE losses. This branch uses foundation and exposure inputs directly, not company forecasts from 08.

## Bridge through the study

Together with 08, this supports a bounded conclusion: extra historical dependence does not establish stable stock-risk forecast value, and portfolio benefits depend on the baseline and holdings. This completes the study; further research is separate.
